# Description
This notebook computes the distance for the records with respect to the origin point for the selected trips of the route.

In [1]:
import json
import math
import matplotlib.pyplot as plt

In [2]:
from pymongo import     MongoClient
con = MongoClient()

In [3]:
def mydistance(a1,b1,a2,b2):
    '''
    input: location attributes corresponding to point 1 and 2. (lat1, lon1, lat2, lon2)
    output: distance between point 1 and point 2
    function: compute distance between two points using haversine formula
    '''    
    R=6371e3
    x1=math.radians(a1)
    y1=math.radians(b1)
    x2=math.radians(a2)
    y2=math.radians(b2)
    delx=x2-x1
    dely=y2-y1
    c=math.sin(delx/2)*math.sin(delx/2)+math.cos(x1)*math.cos(x2)*math.sin(dely/2)*math.sin(dely/2)
    d=2*math.atan2(math.sqrt(c),math.sqrt(1-c))
    e=R*d
    return(e)

In [4]:
#'''
ProjectDataUsed = True
UsedPreTrained = False
UseMongoDB = True
#'''
'''
ProjectDataUsed = True
UsedPreTrained = True
UseMongoDB = False
'''

'\nProjectDataUsed = True\nUsedPreTrained = True\nUseMongoDB = False\n'

In [5]:
RouteName = 'Git_ISCON_PDPU'

In [6]:
TripsInfo=['20_12_2017__18_31_19', '18_01_2018__07_38_10']

In [7]:
if UseMongoDB==True:
    '''Compute the distance for the trip records of the selected trips'''
    for TripIndex in range(len(TripsInfo)):
        LocationRecords = [lr for lr in con[RouteName][TripsInfo[TripIndex]+'.Filtered'].find().sort([('epoch',1)])]
        i=0
        #distanceFromOrigin=0.0
        LocationRecordsReference = [lr for lr in con[RouteName][TripsInfo[TripIndex]+'.Filtered'].find().sort([('epoch',1)])]

        distanceFromOrigin = mydistance(LocationRecordsReference[0]["Latitude"],LocationRecordsReference[0]["Longitude"],LocationRecords[0]["Latitude"],LocationRecords[0]["Longitude"])
        normalizedDistanceFromOrigin=0.0
        #print(distanceFromOrigin)
        #input()
        #totalDistance=0
        totalDistance=distanceFromOrigin
        for index in range(len(LocationRecords)-1):
            lt1=LocationRecords[index]["Latitude"]
            ln1=LocationRecords[index]["Longitude"]
            lt2=LocationRecords[index+1]["Latitude"]
            ln2=LocationRecords[index+1]["Longitude"]
            totalDistance += mydistance(lt1,ln1, lt2,ln2)     

        normalizedDistanceFromOrigin = distanceFromOrigin/totalDistance
        LocationRecords[i]['distanceFromOrigin']=distanceFromOrigin
        LocationRecords[i]['normalizedDistanceFromOrigin']=normalizedDistanceFromOrigin
        for i in range(1, len(LocationRecords)):
            lt1=LocationRecords[i-1]["Latitude"]
            ln1=LocationRecords[i-1]["Longitude"]
            lt2=LocationRecords[i]["Latitude"]
            ln2=LocationRecords[i]["Longitude"]

            distanceFromOrigin +=   mydistance(lt1,ln1,lt2,ln2)
            normalizedDistanceFromOrigin =  distanceFromOrigin/totalDistance

            LocationRecords[i]['distanceFromOrigin']=distanceFromOrigin
            LocationRecords[i]['normalizedDistanceFromOrigin']=normalizedDistanceFromOrigin
            #print(LocationRecords)
            #input()
        #con[RouteName].drop_collection(TripsInfo[TripIndex]+'.LocationRecordsWithDistanceFromOrigin')
        con[RouteName]['TripInfo'].update_one({'SingleTripInfo':TripsInfo[TripIndex]},{'$set':{'totalDistance':totalDistance}})
        con[RouteName][TripsInfo[TripIndex]+'.LocationRecordsWithDistanceFromOrigin'].insert_many(LocationRecords)
